# Rijndael Algorithm - Advanced Encryption Standard (AES)

- AES is a **Block Cipher** i.e it takes a plaintext block of a fixed number of bits and produces a Ciphertext block of the same size.
- AES uses a **block size of 128 bits (16 bytes)**.

AES takes these 16 bytes:

|1 &#8203;|2 &#8203;|3 &#8203;|4 &#8203;|5 &#8203;|6 &#8203;|7 &#8203;|8 &#8203;|9 &#8203;|10|11|12|13|14|15|16|
|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|

Then lays them down in a 4x4 grid:

|1 &#8203;|5 &#8203;|9|13|
|---|---|---|---|
|**2**|**6**|**10**|**14**|
|**3**|**7**|**11**|**15**|
|**4**|**8**|**12**|**16**|

This is known as **the state of the algorithm**. This state is now updated repeatedly in rounds to get our encrypted cipher block. However, the number of rounds depends on the key length:

|N (Number of Rounds)|Key Size (in bits)|
|---|---|
|10|128|
|12|192|
|14|256|

### S-Box (Substitution Box)

The AES S-Box is a fixed, nonlinear byte-substitution table mapping:
$$
\{0,1\}^8\rightarrow \{0,1\}^8
$$
Where, every input byte (0 to 255) is replaced by exactly one output byte. Without the S-Box, AES would be trivially broken using linear algebra. The S-Box provides confusion by making the relationship between key and ciphertext highly nonlinear and complex.

In [ ]:
# AES S-Box lookup table: maps each byte value (0-255) to its substituted byte
SBOX = [
    0x63,0x7c,0x77,0x7b,0xf2,0x6b,0x6f,0xc5,0x30,0x01,0x67,0x2b,0xfe,0xd7,0xab,0x76,
    0xca,0x82,0xc9,0x7d,0xfa,0x59,0x47,0xf0,0xad,0xd4,0xa2,0xaf,0x9c,0xa4,0x72,0xc0,
    0xb7,0xfd,0x93,0x26,0x36,0x3f,0xf7,0xcc,0x34,0xa5,0xe5,0xf1,0x71,0xd8,0x31,0x15,
    0x04,0xc7,0x23,0xc3,0x18,0x96,0x05,0x9a,0x07,0x12,0x80,0xe2,0xeb,0x27,0xb2,0x75,
    0x09,0x83,0x2c,0x1a,0x1b,0x6e,0x5a,0xa0,0x52,0x3b,0xd6,0xb3,0x29,0xe3,0x2f,0x84,
    0x53,0xd1,0x00,0xed,0x20,0xfc,0xb1,0x5b,0x6a,0xcb,0xbe,0x39,0x4a,0x4c,0x58,0xcf,
    0xd0,0xef,0xaa,0xfb,0x43,0x4d,0x33,0x85,0x45,0xf9,0x02,0x7f,0x50,0x3c,0x9f,0xa8,
    0x51,0xa3,0x40,0x8f,0x92,0x9d,0x38,0xf5,0xbc,0xb6,0xda,0x21,0x10,0xff,0xf3,0xd2,
    0xcd,0x0c,0x13,0xec,0x5f,0x97,0x44,0x17,0xc4,0xa7,0x7e,0x3d,0x64,0x5d,0x19,0x73,
    0x60,0x81,0x4f,0xdc,0x22,0x2a,0x90,0x88,0x46,0xee,0xb8,0x14,0xde,0x5e,0x0b,0xdb,
    0xe0,0x32,0x3a,0x0a,0x49,0x06,0x24,0x5c,0xc2,0xd3,0xac,0x62,0x91,0x95,0xe4,0x79,
    0xe7,0xc8,0x37,0x6d,0x8d,0xd5,0x4e,0xa9,0x6c,0x56,0xf4,0xea,0x65,0x7a,0xae,0x08,
    0xba,0x78,0x25,0x2e,0x1c,0xa6,0xb4,0xc6,0xe8,0xdd,0x74,0x1f,0x4b,0xbd,0x8b,0x8a,
    0x70,0x3e,0xb5,0x66,0x48,0x03,0xf6,0x0e,0x61,0x35,0x57,0xb9,0x86,0xc1,0x1d,0x9e,
    0xe1,0xf8,0x98,0x11,0x69,0xd9,0x8e,0x94,0x9b,0x1e,0x87,0xe9,0xce,0x55,0x28,0xdf,
    0x8c,0xa1,0x89,0x0d,0xbf,0xe6,0x42,0x68,0x41,0x99,0x2d,0x0f,0xb0,0x54,0xbb,0x16
]

### RCon (Round Constants)

RCON is a sequence of round constants used only in KeyExpansion.

In [ ]:
# Round constants used during KeyExpansion (only the first byte of each word is non-zero)
RCON = [
    0x01,0x02,0x04,0x08,0x10,
    0x20,0x40,0x80,0x1B,0x36
]

### 1. AddRoundKey

AddRoundKey is the operation where the AES state is combined with a round-specific key using bitwise XOR. Mathematically:

$$
State \leftarrow State \oplus RoundKey_i
$$

- State: 128-bit internal AES state
- RoundKeyᵢ: 128-bit subkey derived from the master key
- Operation: XOR ($\oplus$)

XOR has three critical properties:
- Reversible i.e $(A \oplus K) \oplus K = A$
- No information is lost
- Cheap and deterministic

However! AddRoundKey is the ONLY step where the secret key directly affects the state.

In [ ]:
def add_round_key(state, round_key):
    """XOR the AES `state` with `round_key` in-place and return the state."""
    # `state` and `round_key` are 4x4 matrices of bytes (rows x columns)
    for r in range(4):
        for c in range(4):
            state[r][c] ^= round_key[r][c]
    return state

### 2. KeyExpansion

KeyExpansion is the deterministic algorithm that takes the original AES master key and expands it into multiple round keys, one per AES round (plus one initial key). A Key Schedule algorithm calculates all the round keys from the key. So the initial 128-bit key is used to create $N_r+1$ 128-bit round keys which will be used in the corresponding round of the encryption.

$$
MasterKey \rightarrow \{RoundKey_0, RoundKey_1, RoundKey_2, ... , RoundKey_N\}
$$

Where:
- Each RoundKey is 128 bits
- $N = 10, 12, 14$ for AES-128/192/256 respectively

The Key Expansion routine creates round keys word-by-word, where a word is an array of four bytes. For AES-128 there are 44 words

1. The first four words ($w_0, w_1, w_2, w_3$) are derived from the 16-byte cipher key.

    $$
    \text{Cipher Key} =
    \begin{matrix}
    k_0 & k_1 & k_2 & k_3 & k_4 & k_5 & k_6 & k_7 &
    k_8 & k_9 & k_{10} & k_{11} & k_{12} & k_{13} & k_{14} & k_{15}
    \end{matrix}
    $$

    $$
    w_0 =
    \begin{matrix}
    k_0 & k_1 & k_2 & k_3
    \end{matrix}
    \qquad
    w_1 =
    \begin{matrix}
    k_4 & k_5 & k_6 & k_7
    \end{matrix}
    $$
    $$
    w_2 =
    \begin{matrix}
    k_8 & k_9 & k_{10} & k_{11}
    \end{matrix}
    \qquad
    w_3 =
    \begin{matrix}
    k_{12} & k_{13} & k_{14} & k_{15}
    \end{matrix}
    $$

<br>

2. The rest of the words ($w_i$ for $i$ = 4 to 43) are made as follows:
- If $i$ mod 4 $\neq$ 0, $w_i = w_{i-1} \oplus w_{i-4}$, i.e each word is made from the one at the left and the one at the top.
- If  $i$ mod 4 = 0, $w_i = t \oplus w_{i-4}$. Where $t$ is a temporary word resulting from applying two routines **SubWord** and **RotWord** on $w_{i-1}$ and XORing the result with a round constants, RCon.

$$
t = SubWord(RotWord(w_{i-1}))\oplus RCon_{i/4}
$$

**RotWord** (rotate word) routine takes a word as an array of four bytes and shifts each byte to the left with wrapping.  
**SubWord** (substitute word) routine takes each byte in the word and substitutes another byte for it.  
**RCon** (Round Constant) is a 4-byte value in which the rightmost three bytes are always zero. 

As for AES-192 and AES-256, key expansion is similar:
- In AES-192, the words are generated in groups of six instead of four.
    - The cipher key creates the first six words ($w_0$ to $w_5$)
    - If $i$ mod 6 $\neq$ 0, $w_i \leftarrow w_{i-1} + w_{i-6}$, else $w_i \leftarrow t + w_{i-6}$
- In AES-256, the words are generated in groups of eight instead of four.
    - The cipher key creates the first eight words ($w_0$ to $w_7$)
    - If $i$ mod 8 $\neq$ 0, $w_i \leftarrow w_{i-1} + w_{i-8}$, else $w_i \leftarrow t + w_{i-8}$
    - If $i$ mod 4 = 0, but $i$ mod 8 $\neq$ 0, then $w_i = SubWord(w_{i-1})+w_{i-8}$ 

In [ ]:
def rot_word(word):
    """Rotate a 4-byte word left by one byte."""
    return word[1:] + word[:1]

def sub_word(word):
    """Apply the AES SBOX substitution to each byte in a 4-byte word."""
    return [SBOX[b] for b in word]

def key_expansion(key):
    """Expand a 16-byte AES-128 `key` into (Nr+1) round keys.
    Returns a list of 11 round keys, each round key is a 4x4 byte matrix."""
    Nk = 4
    Nb = 4
    Nr = 10

    # Initialize words from the original 16-byte key (4 words)
    w = [key[i:i+4] for i in range(0, 16, 4)]

    for i in range(Nk, Nb * (Nr + 1)):
        temp = w[i - 1].copy()

        if i % Nk == 0:
            temp = sub_word(rot_word(temp))
            temp[0] ^= RCON[(i // Nk) - 1]

        w.append([w[i - Nk][j] ^ temp[j] for j in range(4)])

    # Group words into round keys (4 words per round key)
    return [w[4*i:4*(i+1)] for i in range(Nr + 1)]

### 3. SubBytes

The first transformation, SubBytes, is used at the encryption site. To substitute a byte, we interpret the byte as two hexadecimal digits. The left digit defines the row and the right digit defines the column of the substitution table. The two hexadecimal digits at the junction of the row and the column are the new byte.

|97|99|32|32|
|---|---|---|---|
|**32**|**114**|**102**|**121**|
|**115**|**101**|**111**|**111**|
|**101**|**116**|**114**|**117**|

to

|239|251|183|183|
|---|---|---|---|
|**183**|**64**|**51**|**182**|
|**143**|**77**|**168**|**168**|
|**77**|**146**|**64**|**157**|

However, even if the substitution is complex, its public knowledge and known in advance. So, encryption only on the substitution isn't reliable. To prevent adversaries from reversing the encryption, AES interleafs AddRoundKey along with SubBytes. In each round, AES substitutes the byte and then adds another AddRoundKey in each round. Individually, SubBytes and AddRoundKey doesn't do much for security. However, combining them together results in more confusion.

In [ ]:
def sub_bytes(state):
    """Substitute each byte in the state using the AES S-Box (in-place)."""
    for r in range(4):
        for c in range(4):
            state[r][c] = SBOX[state[r][c]]
    return state

### 4. ShiftRows

ShiftRows is a cyclic permutation of the rows of the AES state matrix. The number of shifts depends on the row number (0, 1, 2 or 3) of the state matrix. This means the row 0 is not shifted at all and the last row is shifted three bytes. This operation breaks column isolation, forces bytes to move across columns and enables cross-column diffusion.

Suppose state:

|97|99|32|32|
|---|---|---|---|
|**32**|**114**|**102**|**121**|
|**115**|**101**|**111**|**111**|
|**101**|**116**|**114**|**117**|

After ShiftRows:


|97|99|32|32|
|---|---|---|---|
|**114**|**102**|**121**|**32**|
|**111**|**111**|**115**|**101**|
|**117**|**101**|**116**|**114**|

In [ ]:
def shift_rows(state):
    """Cyclically shift rows of the AES state to the left.
    Row 0: no shift, Row 1: 1 byte, Row 2: 2 bytes, Row 3: 3 bytes."""
    state[1] = state[1][1:] + state[1][:1]
    state[2] = state[2][2:] + state[2][:2]
    state[3] = state[3][3:] + state[3][:3]
    return state

### 5. MixColumns

The MixColumns transformation operates at the column level; it transforms each column of the state to a new column. MixColumns transforms each column of the AES state by multiplying it with a fixed matrix over $GF(2^8)$. The $GF(2^8)$ field is the Galois field (finite field) containing exactly $(2^{8}=256)$ elements. It is an algebric method of introducing diffusion to the state.

Each column is treated as a vector of 4 bytes:

$$\begin{bmatrix}
b_0 \\ b_1 \\ b_2 \\ b_3
\end{bmatrix}$$

It is then multiplied by:

$$\begin{bmatrix}
02 & 03 & 01 & 01 \\
01 & 02 & 03 & 01 \\
01 & 01 & 02 & 03 \\
03 & 01 & 01 & 02
\end{bmatrix}$$

$\fbox{Note:}$ This matrix was chosen because it satisfies very specific mathematical properties:
- Maximum diffusion (Maximum Branch Number = 5)
- Invertibility
    $$ Inverse\; Matrix = \begin{bmatrix}
    0E & 0B & 0D & 09 \\
    09 & 0E & 0B & 0D \\
    0D & 09 & 0E & 0B \\
    0B & 0D & 09 & 0E
    \end{bmatrix}$$
- Efficient implementation
- No structural weaknesses

This results in the output byte becoming:
- A combination of all 4 input bytes
- Weighted by small constants (01, 02, 03)

$\fbox{Note:}$ MixColumns is omitted in the final round. This is because it simplifies decryption, maintainss symmetry, no security loss and improves performance. By the final round, every output byte already depends on every input byte and key bit.

MixColumns is used in tandem with ShiftRows because:
- ShiftRows alone:
    - Only moves bytes
    - No value mixing
    - Fully reversible visually
- MixColumns alone:
    - Only mixes within columns
    - No cross-column spread
- Combined:
    - ShiftRows moves bytes across columns
    - MixColumns blends them
    - Next round repeats results in exponential diffusion

In [ ]:
def gmul(a, b):
    """Galois field (2^8) multiplication of `a` by `b`.
    Used to compute MixColumns multiplications (e.g., multiply by 2 or 3)."""
    p = 0
    for _ in range(8):
        if b & 1:
            p ^= a
        hi = a & 0x80
        a = (a << 1) & 0xFF
        if hi:
            a ^= 0x1B
        b >>= 1
    return p

def mix_columns(state):
    """Transform each column of `state` using the fixed MixColumns matrix."""
    for c in range(4):
        col = [state[r][c] for r in range(4)]
        state[0][c] = gmul(col[0],2) ^ gmul(col[1],3) ^ col[2] ^ col[3]
        state[1][c] = col[0] ^ gmul(col[1],2) ^ gmul(col[2],3) ^ col[3]
        state[2][c] = col[0] ^ col[1] ^ gmul(col[2],2) ^ gmul(col[3],3)
        state[3][c] = gmul(col[0],3) ^ col[1] ^ col[2] ^ gmul(col[3],2)
    return state

## Final Algorithm

1. We start with some data we want to encrypt.
2. We then apply a **KeyExpansion** step that takes the key and expands it into many round keys each of which values depend on the original key.
3. We then add in the **AddRoundKey**.
4. Now we start an iteration for each round:
    1. We start by applying a **SubBytes** and substitute each byte with its corrensponding replacement.
    2. Now we **ShiftRows**, shifting each row by a different amount.
    3. Then we **MixColumns**, combining the values from each column to get new column values.
    4. At the end of each round, we **AddRoundKey** one more time.
5. After completing **N** number of rounds, we get our final encrypted result.

In [ ]:
def bytes_to_state(b):
    """Convert 16-byte input into a 4x4 AES state matrix (rows x columns)."""
    return [[b[r + 4*c] for c in range(4)] for r in range(4)]

def state_to_bytes(state):
    """Flatten the 4x4 AES state matrix back into 16 bytes (column-major)."""
    return bytes(state[r][c] for c in range(4) for r in range(4))

def pad(data):
    """PKCS#7-style padding to a 16-byte boundary. Returns padded bytes."""
    pad_len = 16 - (len(data) % 16)
    return data + bytes([pad_len] * pad_len)

def aes_encrypt_block(block, round_keys):
    """Encrypt a single 16-byte block using provided `round_keys`.
    The function performs the standard AES-128 round sequence and returns 16-byte ciphertext."""
    state = bytes_to_state(block)

    # Initial AddRoundKey
    state = add_round_key(state, round_keys[0])

    # Main rounds (1..9)
    for rnd in range(1, 10):
        state = sub_bytes(state)
        state = shift_rows(state)
        state = mix_columns(state)
        state = add_round_key(state, round_keys[rnd])

    # Final round (NO MixColumns)
    state = sub_bytes(state)
    state = shift_rows(state)
    state = add_round_key(state, round_keys[10])

    return state_to_bytes(state)

def aes_encrypt(plaintext: bytes, key: bytes):
    """High-level AES-128 encrypt function.
    - `plaintext`: bytes to encrypt (will be padded),
    - `key`: 16-byte AES key."""
    assert len(key) == 16, "AES-128 requires 16-byte key"

    round_keys = key_expansion(list(key))
    plaintext = pad(plaintext)

    ciphertext = b""
    for i in range(0, len(plaintext), 16):
        block = plaintext[i:i+16]
        ciphertext += aes_encrypt_block(block, round_keys)

    return ciphertext

def main():
    # Example usage with a 16-byte key
    key = b"ThisIsA16ByteKey"

    plaintext = b"This is a Python implementation of AES"
    ciphertext = aes_encrypt(plaintext, key)

    print("Plaintext:", plaintext)
    print("Ciphertext (hex):", ciphertext.hex())

if __name__ == "__main__":
    main()

Plaintext: b'This is a Python implementation of AES'
Ciphertext (hex): 7c972bf9e7f48f52ab4186b11b56f476b89c42fda2edffb1798f60a9f6a39605b7f367d8698c2b618b4ca93fc96f67ed
